# Clase 11 — Construyendo un chatbot (parte 1: la base de datos)

**Objetivos de hoy**
- Entender cómo se estructura una conversación padre-respuesta.
- Filtrar comentarios reales según criterios de calidad.
- Construir la base de datos que alimentará al chatbot.

Consulta el [capítulo 9 del manual](../manual/09_chatbot.md) para la explicación completa.

**Nota**: en lugar de descargar el volcado completo de comentarios de Reddit (varios gigabytes), usamos aquí una pequeña muestra de ejemplo con la misma estructura, para practicar el pipeline completo de limpieza e inserción.

## 1. Una muestra de comentarios en bruto

Cada fila simula un comentario de Reddit: su identificador, el identificador de su padre, el texto del padre, el texto del comentario, el subreddit, la fecha y su puntuación.

In [ ]:
comentarios_en_bruto = [
    {"comment_id": "c1", "parent_id": "p1", "parent": "¿Cuál es la mejor forma de aprender TensorFlow?", "comment": "Empieza por los fundamentos y practica con notebooks pequeños.", "subreddit": "MachineLearning", "score": 12},
    {"comment_id": "c2", "parent_id": "p2", "parent": "¿Qué opináis de este curso?", "comment": "[deleted]", "subreddit": "MachineLearning", "score": 1},
    {"comment_id": "c3", "parent_id": "p3", "parent": "¿Alguien ha entrenado una CNN 3D?", "comment": "", "subreddit": "MachineLearning", "score": 0},
    {"comment_id": "c4", "parent_id": "p4", "parent": "¿RNN o Transformer para este caso?", "comment": "Depende del tamaño del dataset, pero para secuencias cortas una RNN suele bastar.", "subreddit": "MachineLearning", "score": 8},
]

print(f"Comentarios en bruto: {len(comentarios_en_bruto)}")

## 2. Limpia y valida cada comentario

In [ ]:
def limpiar_texto(texto):
    texto = texto.replace("\n", " ").replace("\r", " ")
    texto = texto.replace('"', "'")
    return texto.strip()

def es_aceptable(comentario):
    if len(comentario) < 1 or len(comentario) > 1000:
        return False
    if comentario in ("[deleted]", "[removed]"):
        return False
    return True

comentarios_validos = []
for c in comentarios_en_bruto:
    texto_limpio = limpiar_texto(c["comment"])
    if es_aceptable(texto_limpio):
        c["comment"] = texto_limpio
        comentarios_validos.append(c)

print(f"Comentarios válidos tras filtrar: {len(comentarios_validos)} de {len(comentarios_en_bruto)}")

## 3. Crea la base de datos SQLite con la tabla `parent_reply`

In [ ]:
import sqlite3

conexion = sqlite3.connect("chatbot.db")
cursor = conexion.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS parent_reply (
    parent_id TEXT,
    comment_id TEXT,
    parent TEXT,
    comment TEXT,
    subreddit TEXT,
    score INTEGER
)
""")
conexion.commit()

## 4. Inserta los comentarios válidos

In [ ]:
for c in comentarios_validos:
    cursor.execute(
        "INSERT INTO parent_reply (parent_id, comment_id, parent, comment, subreddit, score) "
        "VALUES (?, ?, ?, ?, ?, ?)",
        (c["parent_id"], c["comment_id"], c["parent"], c["comment"], c["subreddit"], c["score"]),
    )
conexion.commit()

cursor.execute("SELECT parent, comment FROM parent_reply")
for fila in cursor.fetchall():
    print("P:", fila[0])
    print("R:", fila[1])
    print()

## Resumen

- Filtraste comentarios vacíos, eliminados o demasiado largos antes de guardarlos.
- Construiste una base de datos SQLite con pares pregunta-respuesta, lista para convertirse en datos de entrenamiento.
- Con datos reales, este mismo proceso se aplica a millones de filas en lotes, no una a una.

Continúa con el [notebook de práctica de esta clase](../notebooks_practica/clase11_practica_chatbot_datos.ipynb).